In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [6]:
import transformers

In [7]:
import torch
from transformers import pipeline

ner_pipe = pipeline("token-classification",
                model="cahya/bert-base-indonesian-NER",
                aggregation_strategy="simple")




Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pencarian entitas di wikidata

In [8]:
import requests
import json
import os


def searchWikiData(word):
  url = "https://www.wikidata.org/w/api.php"
  params = {
    "action": "wbsearchentities",
    "search": word,
    "language": "id",
    "format": "json"
  }
  headers = {
    "User-Agent": "KAPALM_KG_Builder/1.0 (fadhilra08@gmail.com)"
  }

  try:
    response = requests.get(url, headers=headers, params=params)

    response.raise_for_status()

    data = response.json()

    return data

  except requests.exceptions.RequestException as e:
      return None

CACHE_FILE = "wikidata_cache.json"
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        wiki_cache = json.load(f)
else:
    wiki_cache = {}

def searchWikiData_Cached(word):
    if word in wiki_cache:
        return wiki_cache[word]

    data = searchWikiData(word)

    if data:
        wiki_cache[word] = data
        if len(wiki_cache) % 50 == 0:
            with open(CACHE_FILE, "w") as f:
                json.dump(wiki_cache, f)
    return data


Pengambilan Triplet KG dari id entitas

In [9]:
from SPARQLWrapper import SPARQLWrapper, JSON

def getWikidataKG(entity_id):
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    sparql.agent = "KAPALM_KG_Builder/1.0 (fadhilra08@gmail.com)"

    query = f"""
      SELECT ?propUrl ?propertyLabel ?object ?objectLabel WHERE {{
      VALUES ?propUrl {{ wdt:P31 wdt:P279 wdt:P17 wdt:P131 wdt:P361 wdt:P106 wdt:P69 wdt:P452 }}
      wd:{entity_id} ?propUrl ?object .
      FILTER(isIRI(?object))
      ?property wikibase:directClaim ?propUrl .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "id". }}
    }}
    """
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    try:
        results = sparql.query().convert()
    except Exception as e:
        print(f"SPARQL Error {entity_id}: {e}")
        return []

    triplets = []
    for result in results["results"]["bindings"]:
        subject_id = entity_id

        predicate_code = result["propUrl"]["value"].split('/')[-1]

        predicate_label = result.get("propertyLabel", {}).get("value", predicate_code)

        if result["object"]["type"] == "uri":
            obj_id = result["object"]["value"].split('/')[-1]
            obj_label = result.get("objectLabel", {}).get("value", obj_id)

            triplets.append({
                "subject_id": subject_id,
                "predicate_code": predicate_code,
                "predicate_label": predicate_label,
                "object_id": obj_id,
                "object_label": obj_label
            })
    return triplets

Ubah triplet KG ke Graph

In [11]:

import networkx as nx

def generateGraph(triplets, subject_label):
    G = nx.MultiDiGraph()
    for e in triplets:
        if not G.has_node(e["subject_id"]):
            G.add_node(e["subject_id"], label=subject_label)
        if not G.has_node(e["object_id"]):
            G.add_node(e["object_id"], label=e["object_label"])

        G.add_edge(
            e["subject_id"],
            e["object_id"],
            relation_code=e["predicate_code"],
            relation_label=e["predicate_label"]
        )
    return G

Pengambilan KG triplet (subject, predicate, object) diberi suatu entitas, mengembalikan graf dalam representasi networkX

In [12]:
def retrieveKg(extracted_entities):
    G = nx.MultiDiGraph()

    for ent in extracted_entities:
        word = ent["word"]
        data = searchWikiData_Cached(word)

        if data and "search" in data and len(data["search"]) > 0:
            top_result = data["search"][0]
            entity_id = top_result["id"]

            official_label = top_result.get("label", word)

            triplets = getWikidataKG(entity_id)

            KG = generateGraph(triplets, official_label)
            G = nx.compose(G, KG)

    return G

Mengubah graph networkX menjadi Data pytorch geometric, yang merepresentasikan 1 graf

In [13]:
from torch_geometric.data import Data

def networkXtoData(G, embedding_model, tokenizer, global_rel_map):
    node_list = sorted(list(G.nodes(data=True)))
    node_to_idx = {node[0]: i for i, node in enumerate(node_list)}
    labels = [attr.get('label', str(n_id)) for n_id, attr in node_list]

    with torch.no_grad():
        inputs = tokenizer(labels, padding=True, truncation=True, return_tensors="pt")
        outputs = embedding_model(**inputs)
        node_features = outputs.last_hidden_state[:, 0, :]

    src_list = []
    dst_list = []
    edge_type_list = []

    for src, dst, attr in G.edges(data=True):
        rel_code = attr.get('relation_code', 'related_to')

        if rel_code in global_rel_map:
            src_list.append(node_to_idx[src])
            dst_list.append(node_to_idx[dst])
            edge_type_list.append(global_rel_map[rel_code])
        else:
            src_list.append(node_to_idx[src])
            dst_list.append(node_to_idx[dst])
            edge_type_list.append(global_rel_map['related_to'])

    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
    edge_type = torch.tensor(edge_type_list, dtype=torch.long)

    return Data(x=node_features, edge_index=edge_index, edge_type=edge_type)

In [14]:
from transformers import AutoTokenizer, AutoModel


In [15]:
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
embedding_model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Pengambilan entitas dari konten teks

In [16]:
def extract_entities_from_text(text):
  results = ner_pipe(text[:512])

  entities = []

  valid_labels = ["PER", "ORG", "LOC", "GPE"]

  for item in results:
    if item["entity_group"] in valid_labels:
      entities.append({
          "word": item["word"],
          "label": item["entity_group"]
      })
  return entities

In [17]:
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

global_rel_map = {
    'P31': 0,   # instance of
    'P279': 1,  # subclass of
    'P17': 2,   # country
    'P131': 3,  # located in the administrative territorial entity
    'P361': 4,  # part of
    'P106': 5,  # occupation
    'P69': 6,   # educated at
    'P452': 7,  # industry
    'related_to': 8 # Fallback
}

In [18]:
import time

In [19]:
import torch

import torch.nn as nn

import torch.nn.functional as F


from torch_geometric.nn import GATConv, global_mean_pool
class CoarseGrainedKAPALM(torch.nn.Module):
    def __init__(self, input_node_dim, num_relations, hidden_dim, output_dim, heads=2):
        super().__init__()

        self.node_proj = nn.Linear(input_node_dim, hidden_dim)

        self.edge_emb = nn.Embedding(num_relations, hidden_dim)

        self.conv1 = GATConv(hidden_dim, hidden_dim, heads=heads, dropout=0.6, edge_dim=hidden_dim)
        self.conv2 = GATConv(hidden_dim * heads, output_dim, heads=1, concat=False, dropout=0.6, edge_dim=hidden_dim)
        self.projector = nn.Linear(output_dim, 768)

    def forward(self, data):
        x, edge_index, edge_type = data.x, data.edge_index, data.edge_type
        batch = data.batch

        x = self.node_proj(x)

        edge_attr = self.edge_emb(edge_type)

        x = self.conv1(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = self.conv2(x, edge_index, edge_attr=edge_attr)

        coarse_knowledge = global_mean_pool(x, batch)
        knowledge_vector = self.projector(coarse_knowledge)

        return knowledge_vector

In [20]:

class KAPALM_Classifier(nn.Module):
    def __init__(self, llm_model, gnn_model, num_classes):
        super().__init__()
        self.llm = llm_model
        self.gnn = gnn_model

        self.classifier = nn.Linear(768 + 768, num_classes)

    def forward(self, input_ids, attention_mask, graph_data):
        text_out = self.llm(input_ids=input_ids, attention_mask=attention_mask)
        text_emb = text_out.last_hidden_state[:, 0, :]

        kg_emb = self.gnn(graph_data)

        combined = torch.cat((text_emb, kg_emb), dim=1)

        logits = self.classifier(combined)

        return logits

In [21]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import os
from transformers import AutoTokenizer, AutoModel


tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

llm_model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")
gnn_model = CoarseGrainedKAPALM(input_node_dim=768, num_relations=9, hidden_dim=768, output_dim=768)
model = KAPALM_Classifier(llm_model, gnn_model, num_classes=1)


Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [23]:
from transformers import AutoTokenizer
import torch

from huggingface_hub import hf_hub_download

weights_path = hf_hub_download(repo_id="admirablemount82/KAPALM_NO_PRUNING", filename="kapalm_weights.bin")
model.load_state_dict(torch.load(weights_path))

tokenizer = AutoTokenizer.from_pretrained("admirablemount82/KAPALM_NO_PRUNING")

model.eval()

kapalm_weights.bin:   0%|          | 0.00/519M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

KAPALM_Classifier(
  (llm): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_

In [29]:
model.to(device)

KAPALM_Classifier(
  (llm): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_

In [30]:
import torch
from torch_geometric.data import Batch

def predict_hoax(text, model, tokenizer, device='cuda'):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)


    entities = extract_entities_from_text(text)

    nx_graph = retrieveKg(entities)

    if nx_graph.number_of_nodes() == 0:
        nx_graph.add_node("dummy", label="tidak_diketahui")
        nx_graph.add_edge("dummy", "dummy", relation_code="related_to")

    pyg_data = networkXtoData(nx_graph, embedding_model, tokenizer, global_rel_map)

    batch_graph = Batch.from_data_list([pyg_data]).to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask, batch_graph)
        probability = torch.sigmoid(logits).item()

    return probability

In [32]:

text_hoax = "Pemerintah resmi menjual seluruh pulau Kalimantan kepada China untuk membayar utang negara."
prob_hoax = predict_hoax(text_hoax, model, tokenizer)

print(f"\nText: {text_hoax}")
print(f"Hoax Probability: {prob_hoax:.4f}")
print(f"Prediction: {'HOAX' if prob_hoax > 0.5 else 'FACT'}")


Text: Pemerintah resmi menjual seluruh pulau Kalimantan kepada China untuk membayar utang negara.
Hoax Probability: 0.9904
Prediction: HOAX
